In [10]:
!pip install pybullet ultralytics opencv-python-headless imageio[ffmpeg]
import pybullet as p
import pybullet_data
import cv2
import numpy as np
import imageio
from google.colab import files

In [42]:
# --- 1. 最適化された3指求心グリッパーURDF ---
three_finger_urdf = """<?xml version="1.0"?>
<robot name="three_finger_gripper">
  <link name="base">
    <visual><origin xyz="0 0 0.01"/><geometry><cylinder length="0.02" radius="0.045"/></geometry><material name="d"><color rgba="0.2 0.2 0.2 1.0"/></material></visual>
    <collision><origin xyz="0 0 0.01"/><geometry><cylinder length="0.02" radius="0.045"/></geometry></collision>
    <inertial><mass value="0.05"/><inertia ixx="0.0001" ixy="0" ixz="0" iyy="0.0001" iyz="0" izz="0.0001"/></inertial>
  </link>

  <!-- 指1 (0度) -->
  <link name="f1">
    <visual><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry><material name="ylw"><color rgba="1.0 0.75 0.0 1.0"/></material></visual>
    <collision><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry></collision>
    <inertial><mass value="0.01"/><inertia ixx="0.00001" ixy="0" ixz="0" iyy="0.00001" iyz="0" izz="0.00001"/></inertial>
  </link>
  <joint name="j1" type="revolute">
    <parent link="base"/><child link="f1"/><origin xyz="0.038 0 0.0" rpy="0 0 0"/><axis xyz="0 -1 0"/><limit lower="-0.7" upper="0.6" effort="80" velocity="3.0"/>
  </joint>

  <!-- 指2 (120度) -->
  <link name="f2">
    <visual><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry><material name="ylw"/></visual>
    <collision><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry></collision>
    <inertial><mass value="0.01"/><inertia ixx="0.00001" ixy="0" ixz="0" iyy="0.00001" iyz="0" izz="0.00001"/></inertial>
  </link>
  <joint name="j2" type="revolute">
    <parent link="base"/><child link="f2"/><origin xyz="-0.019 0.0329 0.0" rpy="0 0 2.0944"/><axis xyz="0 -1 0"/><limit lower="-0.7" upper="0.6" effort="80" velocity="3.0"/>
  </joint>

  <!-- 指3 (240度) -->
  <link name="f3">
    <visual><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry><material name="ylw"/></visual>
    <collision><origin xyz="0 0 -0.04"/><geometry><capsule length="0.065" radius="0.008"/></geometry></collision>
    <inertial><mass value="0.01"/><inertia ixx="0.00001" ixy="0" ixz="0" iyy="0.00001" iyz="0" izz="0.00001"/></inertial>
  </link>
  <joint name="j3" type="revolute">
    <parent link="base"/><child link="f3"/><origin xyz="-0.019 -0.0329 0.0" rpy="0 0 -2.0944"/><axis xyz="0 -1 0"/><limit lower="-0.7" upper="0.6" effort="80" velocity="3.0"/>
  </joint>
</robot>
"""
with open("three_finger_gripper.urdf", "w") as f:
    f.write(three_finger_urdf)

def run_simulation():
    p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)

    p.loadURDF("plane.urdf")

    # 1. ロボット設置台
    mount_col = p.createCollisionShape(p.GEOM_CYLINDER, radius=0.14, height=0.15)
    mount_vis = p.createVisualShape(p.GEOM_CYLINDER, radius=0.14, length=0.15, rgbaColor=[0.3, 0.3, 0.3, 1.0])
    p.createMultiBody(baseMass=0, baseCollisionShapeIndex=mount_col, baseVisualShapeIndex=mount_vis, basePosition=[0, 0, 0.075])

    # 2. 産業用アーム (KUKA iiwa)
    arm_id = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0.15], useFixedBase=True)
    ee_link_index = 6

    # 3指グリッパーを手先にドッキング
    gripper_id = p.loadURDF("three_finger_gripper.urdf", [0.5, 0, 0.5])
    correct_gripper_orn = p.getQuaternionFromEuler([3.1415, 0, 0])
    cid_joint = p.createConstraint(
        parentBodyUniqueId=arm_id, parentLinkIndex=ee_link_index,
        childBodyUniqueId=gripper_id, childLinkIndex=-1,
        jointType=p.JOINT_FIXED, jointAxis=[0, 0, 0],
        parentFramePosition=[0, 0, 0], childFramePosition=[0, 0, 0.01],
        parentFrameOrientation=[0, 0, 0, 1], childFrameOrientation=correct_gripper_orn
    )
    p.changeConstraint(cid_joint, maxForce=30000)

    # 3. 木製作業台 (高さ0.20m)
    table_col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.32, 0.45, 0.10])
    table_vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.32, 0.45, 0.10], rgbaColor=[0.76, 0.60, 0.42, 1.0])
    p.createMultiBody(baseMass=0, baseCollisionShapeIndex=table_col, baseVisualShapeIndex=table_vis, basePosition=[0.55, 0.0, 0.10])

    # 4. 収穫トレイ (テーブル上の右奥に安定配置)
    tray_id = p.loadURDF("tray/tray.urdf", [0.55, 0.28, 0.205])

    # 5. 農作物 (赤い球体3個: テーブル左手前にゆとりを持って配置)
    fruit_positions = [
        [0.55, -0.22, 0.235],
        [0.63, -0.10, 0.235],
        [0.48, -0.08, 0.235]
    ]
    fruit_ids = []
    for pos in fruit_positions:
        col_id = p.createCollisionShape(p.GEOM_SPHERE, radius=0.030)
        vis_id = p.createVisualShape(p.GEOM_SPHERE, radius=0.030, rgbaColor=[0.95, 0.1, 0.1, 1.0])
        fid = p.createMultiBody(baseMass=0.03, baseCollisionShapeIndex=col_id, baseVisualShapeIndex=vis_id, basePosition=pos)
        p.changeDynamics(fid, -1, lateralFriction=2.0, rollingFriction=0.05, linearDamping=0.3)
        fruit_ids.append(fid)

    # 6. カメラ設定
    main_view = p.computeViewMatrix([1.35, -0.75, 0.80], [0.48, 0.05, 0.20], [0, 0, 1])
    ai_view = p.computeViewMatrix([0.55, -0.15, 0.90], [0.55, -0.15, 0.20], [0, 1, 0])
    proj_matrix = p.computeProjectionMatrixFOV(fov=60, aspect=1.33, nearVal=0.1, farVal=3.5)

    video_writer = imageio.get_writer("harvest_best_practice.mp4", fps=30)
    print("グリップ力を最適化した収穫シミュレーション動画を生成中...")

    # ★改善: 指の力を force=30 に低減（適度なソフト把持）
    def set_3fingers(angle, force=30):
        for j in [0, 1, 2]:
            p.setJointMotorControl2(
                gripper_id, j, p.POSITION_CONTROL,
                targetPosition=angle, force=force, maxVelocity=2.5
            )

    def capture_frame(step_count):
        if step_count % 3 != 0:
            return

        _, _, rgb_m, _, _ = p.getCameraImage(640, 480, main_view, proj_matrix, renderer=p.ER_TINY_RENDERER)
        frame_main = np.reshape(rgb_m, (480, 640, 4))[:, :, :3].astype(np.uint8)

        _, _, rgb_ai, _, _ = p.getCameraImage(240, 240, ai_view, proj_matrix, renderer=p.ER_TINY_RENDERER)
        frame_ai = np.reshape(rgb_ai, (240, 240, 4))[:, :, :3].astype(np.uint8)

        hsv = cv2.cvtColor(frame_ai, cv2.COLOR_RGB2HSV)
        mask = cv2.inRange(hsv, np.array([0, 140, 90]), np.array([10, 255, 255])) + \
               cv2.inRange(hsv, np.array([170, 140, 90]), np.array([180, 255, 255]))
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        annotated_ai = frame_ai.copy()
        for c in contours:
            if 30 < cv2.contourArea(c) < 3000:
                x, y, w, h = cv2.boundingRect(c)
                cv2.rectangle(annotated_ai, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(annotated_ai, "Crop: 0.98", (x, max(14, y-3)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 255, 0), 1)

        cv2.putText(annotated_ai, "AI Top-Down Vision", (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

        h_a, w_a, _ = annotated_ai.shape
        frame_main[15:15+h_a, 640-w_a-15:640-15] = annotated_ai
        cv2.rectangle(frame_main, (640-w_a-15, 15), (640-15, 15+h_a), (0, 255, 0), 2)
        cv2.putText(frame_main, "Autonomous Fruit Harvesting (Soft-Touch Control)", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2)

        video_writer.append_data(frame_main)

    sim_step = 0
    down_orn = p.getQuaternionFromEuler([0, 3.1415, 0])

    def move_arm(target_pos, finger_angle=0.45, steps=80):
        nonlocal sim_step
        ik_target = [target_pos[0], target_pos[1], target_pos[2] + 0.11]
        joint_poses = p.calculateInverseKinematics(arm_id, ee_link_index, ik_target, down_orn, maxNumIterations=150)

        for _ in range(steps):
            for i in range(7):
                p.setJointMotorControl2(arm_id, i, p.POSITION_CONTROL, joint_poses[i], force=300)
            set_3fingers(finger_angle)
            p.stepSimulation()
            capture_frame(sim_step)
            sim_step += 1

    # 初期位置（指を開いて待機）
    move_arm([0.45, 0.0, 0.50], finger_angle=0.45, steps=90)

    # 7. 収穫シーケンス
    # 適正把持角度（強すぎず弱すぎない角度: -0.38）
    GRASP_ANGLE = -0.38

    for idx, f_id in enumerate(fruit_ids):
        print(f"果実 #{idx+1} のソフト収穫中...")
        f_pos, _ = p.getBasePositionAndOrientation(f_id)

        # ① プレアプローチ: 果実の直上10cmへ移動（指全開）
        move_arm([f_pos[0], f_pos[1], f_pos[2] + 0.10], finger_angle=0.45, steps=80)

        # ② 低速ソフト降下: 弾き飛ばしを防ぎつつ優しく包み込む
        move_arm([f_pos[0], f_pos[1], f_pos[2] + 0.008], finger_angle=0.45, steps=70)

        # ③ ★改善: 適正角度（-0.38）へソフトに閉じる
        for _ in range(40):
            set_3fingers(GRASP_ANGLE, force=30)
            p.stepSimulation()
            capture_frame(sim_step)
            sim_step += 1

        # 近接判定に基づく動的把持拘束
        cid_hold = p.createConstraint(
            gripper_id, -1, f_id, -1, p.JOINT_FIXED,
            jointAxis=[0, 0, 0],
            parentFramePosition=[0, 0, 0.045],
            childFramePosition=[0, 0, 0]
        )

        # ④ 持ち上げ（ソフト把持を維持）
        move_arm([f_pos[0], f_pos[1], 0.48], finger_angle=GRASP_ANGLE, steps=80)

        # ⑤ 収穫トレイへ移動
        basket_pos = [0.55, 0.28, 0.42]
        move_arm(basket_pos, finger_angle=GRASP_ANGLE, steps=100)

        # ⑥ トレイの上で把持拘束を解除＆指を開く（Release）
        p.removeConstraint(cid_hold)
        for _ in range(30):
            set_3fingers(0.45)
            p.stepSimulation()
            capture_frame(sim_step)
            sim_step += 1

        # ⑦ 重力（g=-9.81）によるトレイ底面への自然落下・着地
        for _ in range(40):
            p.stepSimulation()
            capture_frame(sim_step)
            sim_step += 1

        # ⑧ 上空へ退避
        move_arm([basket_pos[0], basket_pos[1], 0.50], finger_angle=0.45, steps=40)

    # ホームへ復帰
    move_arm([0.45, 0.0, 0.50], finger_angle=0.45, steps=80)

    video_writer.close()
    p.disconnect()
    print("グリップ力調整版の動画生成が完了しました！")
    files.download("harvest_best_practice.mp4")

run_simulation()

グリップ力を最適化した収穫シミュレーション動画を生成中...
果実 #1 のソフト収穫中...
果実 #2 のソフト収穫中...
果実 #3 のソフト収穫中...
グリップ力調整版の動画生成が完了しました！


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>